# Lesson 02: Understanding Foundation Models — How Models Are Built

## Learning Objectives
- Understand the "probabilistic nature" of LLMs: why asking the same question twice may give different answers
- Hands-on tuning of Temperature, Top-P, and other parameters to see their effects
- Trigger and identify AI "hallucinations" (convincing-sounding nonsense)
- Verify that even temperature=0 does not guarantee two identical answers
>
> Comparing models of different sizes belongs to the concept track (see "Experience 3: Model Size Comparison" in the course guide).

> All code cells include detailed comments, suitable for learners with no programming experience.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Tune the Model's "Creativity" — Temperature Experiment

### Activity Goal
Temperature is the core parameter controlling AI output creativity:
- temperature=0: most conservative, answers are nearly identical each time
- temperature=1: creative
- temperature=1.5+: starts getting "wild", may produce strange content

Let's test the same prompt at different temperatures and feel the difference firsthand.

In [ ]:
prompt = 'Write a short four-line poem about autumn.'

# Test at different temperatures
for temp in [0, 0.3, 0.7, 1.0, 1.5]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temp
    )
    print(f'\nTemperature = {temp}')
    print('-' * 30)
    print(response.choices[0].message.content)
    print()

### Discussion

- Is the temperature=0 poem the most "safe" and conventional?
- Does the temperature=1.5 poem bring more "surprises" (or shocks)?
- When should you use low temperature? When is high temperature appropriate?

> Temperature is like AI's "adventurousness" — lower is more conservative, higher is more daring.

In [ ]:
# Bonus experiment: At temperature=0, are two responses exactly the same?
print('Testing determinism at Temperature=0:')
q = 'Summarize the definition of machine learning in one sentence.'

r1 = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':q}], temperature=0)
r2 = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':q}], temperature=0)

print(f'First:  {r1.choices[0].message.content}')
print(f'Second: {r2.choices[0].message.content}')
if r1.choices[0].message.content == r2.choices[0].message.content:
    print('\nBoth responses are identical! Temperature=0 makes the model more deterministic.')
else:
    print('\nResponses are not identical (in practice, temperature=0 is not always absolutely deterministic).')

---

## Activity 2: Trigger and Identify AI "Hallucinations"

### Activity Goal
AI sometimes "convincingly talks nonsense" — this is called hallucination.
It's not a bug; it's a natural result of AI being a "text generator": its goal is to produce plausible text, not factually correct text.

In [ ]:
# Test 1: Ask about a non-existent chapter
print('[Test 1: Non-existent book chapter]')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'List the key content of Chapter 15 of the book "AI Engineering". The book actually only has 10 chapters.'}],
    temperature=0.3)
print(r.choices[0].message.content)
print('\nTruth: This book only has 10 chapters! If AI made something up, that is hallucination.')

# Test 2: Non-existent paper
print('\n[Test 2: Non-existent paper]')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'Please review a 2024 paper titled "Neural Quantum Entanglement in LLMs". I made this paper up — it does not exist.'}],
    temperature=0.3)
print(r.choices[0].message.content)
print('\nTruth: This paper does not exist at all!')

# Test 3: Nudge AI to admit it doesn't know
print('\n[Test 3: Nudging AI to say "I don\'t know"]')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'Which city will host the 2027 Olympics? If you are not sure, just say you don\'t know.'}],
    temperature=0)
print(r.choices[0].message.content)
print('\nIn reality, the 2028 Olympics will be in Los Angeles; there are no 2027 Olympics.')

### Discussion

- In which scenario is AI most prone to hallucination? Why?
- How can you make AI more honestly admit "I don't know"?
- Think of AI as a "very good storyteller friend" — what they say sounds convincing, but you need to verify the facts yourself.

---

## Activity 3: Experience Top-P — Another Creativity Knob

### Activity Goal
Top-P (nucleus sampling) limits AI to choose only from words whose cumulative probability reaches P.
- Top-P=0.1: only choose from the most probable 10% of words — very conservative
- Top-P=0.9: choose from 90% of words — more creative

In [ ]:
prompt = 'Describe the future of artificial intelligence in one sentence.'

for top_p in [0.1, 0.5, 0.9]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content':prompt}],
        top_p=top_p,
        temperature=0.7
    )
    print(f'\nTop-P = {top_p}')
    print('-' * 40)
    print(r.choices[0].message.content)

print('\nSmaller Top-P -> narrower word choices -> more predictable responses')
print('Larger Top-P -> wider word choices -> more diverse responses')

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Temperature tuning | Understand how temperature affects creativity and determinism |
| Top-P tuning | Understand how nucleus sampling limits word choice range |
| Identifying hallucinations | Learn to trigger and recognize when AI is "making things up" |
| Probabilistic nature | Understand why the same question can get different answers |

### Homework
1. Try temperature=2.0 (maximum) and observe whether the output becomes completely incomprehensible
2. Search for "ChatGPT hallucinations examples" to see what interesting hallucinations others have encountered
3. Think: in your own work, if AI hallucinates, what consequences could it cause?